# MENTORA — Train M2: Adaptive Question Generator (FLAN-T5-Large + LoRA)

Do this one last — heaviest lift. Target: **BLEU-4 >= 0.35, ROUGE-L >= 0.50**.

**Note:** our `flan_t5_training_data.csv` currently has 70 prompt/target pairs (target is 500+ — see datasets/SOURCES.md; it scales automatically as M1's question bank grows). Expect this run to be a pipeline smoke test until then.

## 0. Shared setup — mount Drive, confirm GPU, install deps
Run these three cells first in every notebook (per Phase 4 §0).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

In [ ]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

# NOTE: upload the repo's datasets/ folder to Google Drive at this path first
# (mentora_data/{raw,processed,labeled}/...) — see datasets/README.md.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121  # Colab GPU wheel — do NOT reuse the CPU wheel from the backend's laptop plan
!pip install -q transformers peft accelerate sentence-transformers datasets evaluate seqeval spacy sacrebleu rouge_score

### Optional — Weights & Biases logging
Nice for live loss curves / a thesis screenshot. Skip entirely if you'd rather keep things simple — nothing below strictly needs it.

In [ ]:
# !pip install -q wandb
# import wandb
# wandb.login()  # paste your free-tier API key when prompted, once per session

## 1. Load base model + apply LoRA

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model, TaskType
import pandas as pd, numpy as np, glob
from datasets import Dataset

tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-large')
base_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-large')

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q', 'v'],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 2. Load and tokenize the Phase 3 data

In [ ]:
df = pd.read_csv(f'{DATA}/processed/m2/flan_t5_training_data.csv')
ds = Dataset.from_pandas(df).train_test_split(test_size=0.1, seed=42)

def tokenize(batch):
    inputs = tokenizer(batch['input_text'], truncation=True, padding='max_length', max_length=96)
    targets = tokenizer(batch['target_text'], truncation=True, padding='max_length', max_length=256)
    inputs['labels'] = targets['input_ids']
    return inputs

ds = ds.map(tokenize, batched=True)

## 3. Metrics (BLEU-4 + ROUGE-L)

In [ ]:
import evaluate
sacrebleu = evaluate.load('sacrebleu')
rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu = sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    rougeL = rouge.compute(predictions=decoded_preds, references=decoded_labels)['rougeL']
    return {'bleu4': bleu['score'], 'rougeL': rougeL}

## 4. Training arguments — memory-conscious for a free T4

In [ ]:
OUT_DIR = f'{MODELS}/model2_question_generator'
args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1e-4,
    predict_with_generate=True,
    fp16=True,
    save_strategy='steps',
    save_steps=200,
    eval_strategy='steps',
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    report_to='wandb' if 'wandb' in dir() else 'none',
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['test'],
    compute_metrics=compute_metrics,
)

existing = sorted(glob.glob(f'{OUT_DIR}/checkpoint-*'), key=lambda p: int(p.split('-')[-1]))
trainer.train(resume_from_checkpoint=existing[-1] if existing else None)

## 5. Save — LoRA adapter only

In [ ]:
model.save_pretrained(f'{OUT_DIR}/lora_adapter')
tokenizer.save_pretrained(f'{OUT_DIR}/lora_adapter')
print(trainer.evaluate())

## Target metric: BLEU-4 >= 0.35, ROUGE-L >= 0.50

If it plateaus lower:
- Check `target_text` JSON is consistently formatted (Phase 3's script already validates this on generation)
- Bump `r=16` in the LoRA config
- Highest-leverage fix: grow M1's question bank (M2 reuses it directly)